# ARC-AGI-2 Hybrid SOTA (Tier 1 Symbolic Filter + Tier 2 Qwen 4B rsLoRA TTT)

Architecture:
1. **Tier 1 (Instant Symbolic Program Induction):** Evaluates 1- and 2-step DSL primitives (connected components, enclosed region fills, dot connections, periodic room repairs, and geometric symmetries). If 100% verified on train demonstrations, pre-empts attempt_1.
2. **Tier 2 (Test-Time Training Transduction):** 4B Qwen pre-trained model with 128 augmentations, high-rank rsLoRA (r=256), and GPU-accelerated constrained turbo_dfs.
3. **Monotonic Merging:** Ensures every verified task improves accuracy without regression on unsolved tasks.


In [ ]:
# Keep the original global 10-minute submission/write buffer.
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
# Preserve the baseline environment workaround.
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_dsl.py
# Combined ARC DSL primitives, search, and solver
import copy
import numpy as np
from collections import Counter, deque
from dataclasses import dataclass, field
from typing import Any, Callable, Sequence, Union

Grid = list[list[int]]


def grid_equal(g1, g2):
    if g1 is None or g2 is None:
        return False
    a1 = np.asarray(g1, dtype=int)
    a2 = np.asarray(g2, dtype=int)
    return a1.shape == a2.shape and bool(np.array_equal(a1, a2))

"""Domain-Specific Language (DSL) primitives and object perception engine for ARC-AGI."""


from collections import Counter, deque
from dataclasses import dataclass, field
from typing import Sequence
import numpy as np



def to_grid(grid: Grid | np.ndarray) -> Grid:
    """Converts a NumPy array or list-based grid to standard 2D list of ints."""
    if isinstance(grid, np.ndarray):
        return grid.astype(int).tolist()
    return [list(row) for row in grid]


def to_array(grid: Grid | np.ndarray) -> np.ndarray:
    """Converts a 2D grid to a NumPy int32 array."""
    if isinstance(grid, np.ndarray):
        return grid.astype(np.int32)
    return np.array(grid, dtype=np.int32)


@dataclass
class Object:
    """Structured representation of a connected component or object in an ARC grid."""

    coords: set[tuple[int, int]]
    color: int
    bbox: tuple[int, int, int, int]  # (min_r, min_c, max_r, max_c)
    height: int
    width: int
    size: int
    colors: set[int] = field(default_factory=set)

    def crop_from(self, grid: Grid | np.ndarray) -> Grid:
        """Crops this object's bounding box from the given grid."""
        return crop_to_bounding_box(grid, self.bbox)


def find_connected_components(
    grid: Grid | np.ndarray,
    background: int | None = 0,
    connectivity: int = 4,
    monochromatic: bool = True,
) -> list[Object]:
    """Extracts connected components from a grid.

    Args:
        grid: 2D grid of integers.
        background: Pixel value considered background to ignore (None to include all).
        connectivity: 4 for orthogonal neighbors, 8 to include diagonals.
        monochromatic: If True, components are partitioned by identical color.
                       If False, adjacent non-background pixels form a multi-color object.

    Returns:
        List of Object instances sorted predictably by (min_r, min_c).
    """
    arr = to_array(grid)
    h, w = arr.shape
    visited = np.zeros((h, w), dtype=bool)
    objects: list[Object] = []

    if connectivity == 8:
        deltas = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    else:
        deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    for r in range(h):
        for c in range(w):
            val = int(arr[r, c])
            if visited[r, c] or (background is not None and val == background):
                continue

            comp_coords: set[tuple[int, int]] = set()
            comp_colors: set[int] = set()
            queue = deque([(r, c)])
            visited[r, c] = True

            while queue:
                curr_r, curr_c = queue.popleft()
                curr_val = int(arr[curr_r, curr_c])
                comp_coords.add((curr_r, curr_c))
                comp_colors.add(curr_val)

                for dr, dc in deltas:
                    nr, nc = curr_r + dr, curr_c + dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc]:
                        nval = int(arr[nr, nc])
                        if background is not None and nval == background:
                            continue
                        if monochromatic and nval != val:
                            continue
                        visited[nr, nc] = True
                        queue.append((nr, nc))

            min_r = min(cr for cr, _ in comp_coords)
            max_r = max(cr for cr, _ in comp_coords)
            min_c = min(cc for _, cc in comp_coords)
            max_c = max(cc for _, cc in comp_coords)
            obj_h = max_r - min_r + 1
            obj_w = max_c - min_c + 1

            dominant_color = val if monochromatic else Counter(
                arr[cr, cc] for cr, cc in comp_coords
            ).most_common(1)[0][0]

            objects.append(
                Object(
                    coords=comp_coords,
                    color=int(dominant_color),
                    bbox=(min_r, min_c, max_r, max_c),
                    height=obj_h,
                    width=obj_w,
                    size=len(comp_coords),
                    colors=comp_colors,
                )
            )

    objects.sort(key=lambda o: (o.bbox[0], o.bbox[1]))
    return objects


def find_multicolor_objects(
    grid: Grid | np.ndarray,
    background: int | None = 0,
    connectivity: int = 4,
) -> list[Object]:
    """Finds contiguous multi-colored components excluding background."""
    return find_connected_components(
        grid, background=background, connectivity=connectivity, monochromatic=False
    )


def crop_to_bounding_box(
    grid: Grid | np.ndarray,
    bbox: tuple[int, int, int, int],
) -> Grid:
    """Crops the grid to the specified bounding box (min_r, min_c, max_r, max_c)."""
    arr = to_array(grid)
    min_r, min_c, max_r, max_c = bbox
    min_r = max(0, min_r)
    min_c = max(0, min_c)
    max_r = min(arr.shape[0] - 1, max_r)
    max_c = min(arr.shape[1] - 1, max_c)
    if min_r > max_r or min_c > max_c:
        return [[]]
    return arr[min_r : max_r + 1, min_c : max_c + 1].tolist()


def crop_to_content(
    grid: Grid | np.ndarray,
    background: int = 0,
) -> Grid:
    """Crops to the bounding box containing all non-background pixels."""
    arr = to_array(grid)
    mask = arr != background
    if not np.any(mask):
        return [[int(arr[0, 0])]] if arr.size > 0 else [[]]
    rows, cols = np.where(mask)
    bbox = (int(rows.min()), int(cols.min()), int(rows.max()), int(cols.max()))
    return crop_to_bounding_box(arr, bbox)


def pad_grid(
    grid: Grid | np.ndarray,
    target_h: int,
    target_w: int,
    pad_val: int = 0,
    align: str = "top_left",
) -> Grid:
    """Pads grid with pad_val to reach target_h x target_w."""
    arr = to_array(grid)
    curr_h, curr_w = arr.shape
    new_h = max(curr_h, target_h)
    new_w = max(curr_w, target_w)
    out = np.full((new_h, new_w), pad_val, dtype=np.int32)

    if align == "center":
        start_r = (new_h - curr_h) // 2
        start_c = (new_w - curr_w) // 2
    else:  # top_left
        start_r = 0
        start_c = 0

    out[start_r : start_r + curr_h, start_c : start_c + curr_w] = arr
    return out.tolist()


def scale_grid(grid: Grid | np.ndarray, factor: int) -> Grid:
    """Scales grid by integer factor using nearest-neighbor / block repeat."""
    if factor <= 1:
        return to_grid(grid)
    arr = to_array(grid)
    repeated = np.repeat(np.repeat(arr, factor, axis=0), factor, axis=1)
    return repeated.tolist()


def gravity(
    grid: Grid | np.ndarray,
    direction: str = "down",
    obstacle_colors: Sequence[int] | set[int] | None = None,
    background: int = 0,
) -> Grid:
    """Simulates gravity drop in specified direction ('down', 'up', 'left', 'right').

    Non-background cells move in the given direction until blocked by grid boundary,
    an obstacle pixel, or another settled pixel.
    """
    arr = to_array(grid)
    obstacles = set(obstacle_colors) if obstacle_colors is not None else set()

    if direction == "up":
        flipped = np.flipud(arr)
        res = gravity(flipped, direction="down", obstacle_colors=obstacles, background=background)
        return np.flipud(to_array(res)).tolist()

    if direction == "right":
        trans = np.transpose(arr)
        res = gravity(trans, direction="down", obstacle_colors=obstacles, background=background)
        return np.transpose(to_array(res)).tolist()

    if direction == "left":
        trans = np.transpose(arr)
        flipped = np.flipud(trans)
        res = gravity(flipped, direction="down", obstacle_colors=obstacles, background=background)
        unflipped = np.flipud(to_array(res))
        return np.transpose(unflipped).tolist()

    # direction == 'down'
    h, w = arr.shape
    out = np.full((h, w), background, dtype=np.int32)

    for c in range(w):
        col = arr[:, c].tolist()
        for r_idx, val in enumerate(col):
            if val in obstacles:
                out[r_idx, c] = val

        write_r = h - 1
        for r_idx in range(h - 1, -1, -1):
            val = col[r_idx]
            if val in obstacles:
                write_r = r_idx - 1
            elif val != background:
                while write_r >= 0 and out[write_r, c] != background:
                    write_r -= 1
                if write_r >= 0:
                    out[write_r, c] = val
                    write_r -= 1

    return out.tolist()


def flood_fill(
    grid: Grid | np.ndarray,
    r: int,
    c: int,
    new_color: int,
    connectivity: int = 4,
) -> Grid:
    """Flood-fills from (r, c) replacing contiguous target color with new_color."""
    arr = to_array(grid).copy()
    h, w = arr.shape
    if not (0 <= r < h and 0 <= c < w):
        return arr.tolist()

    target_color = int(arr[r, c])
    if target_color == new_color:
        return arr.tolist()

    if connectivity == 8:
        deltas = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]
    else:
        deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    queue = deque([(r, c)])
    arr[r, c] = new_color

    while queue:
        curr_r, curr_c = queue.popleft()
        for dr, dc in deltas:
            nr, nc = curr_r + dr, curr_c + dc
            if 0 <= nr < h and 0 <= nc < w and arr[nr, nc] == target_color:
                arr[nr, nc] = new_color
                queue.append((nr, nc))

    return arr.tolist()


def rotate_90(grid: Grid | np.ndarray) -> Grid:
    """Rotates grid 90 degrees clockwise."""
    return np.rot90(to_array(grid), -1).tolist()


def rotate_180(grid: Grid | np.ndarray) -> Grid:
    """Rotates grid 180 degrees."""
    return np.rot90(to_array(grid), 2).tolist()


def rotate_270(grid: Grid | np.ndarray) -> Grid:
    """Rotates grid 270 degrees clockwise (90 counter-clockwise)."""
    return np.rot90(to_array(grid), 1).tolist()


def reflect_horizontal(grid: Grid | np.ndarray) -> Grid:
    """Reflects grid across the horizontal axis (vertical flip)."""
    return np.flipud(to_array(grid)).tolist()


def reflect_vertical(grid: Grid | np.ndarray) -> Grid:
    """Reflects grid across the vertical axis (horizontal mirror)."""
    return np.fliplr(to_array(grid)).tolist()


def transpose(grid: Grid | np.ndarray) -> Grid:
    """Transposes the grid (rows become columns)."""
    return np.transpose(to_array(grid)).tolist()


def detect_symmetry(grid: Grid | np.ndarray) -> dict[str, bool]:
    """Detects various axes of symmetry in the grid."""
    arr = to_array(grid)
    h, w = arr.shape
    horiz = bool(np.array_equal(arr, np.flipud(arr)))
    vert = bool(np.array_equal(arr, np.fliplr(arr)))
    diag = bool(np.array_equal(arr, arr.T)) if h == w else False
    anti_diag = bool(np.array_equal(arr, np.rot90(arr.T, 2))) if h == w else False

    return {
        "horizontal": horiz,
        "vertical": vert,
        "diagonal": diag,
        "anti_diagonal": anti_diag,
    }


def mirror_append(
    grid: Grid | np.ndarray,
    axis: str = "vertical",
) -> Grid:
    """Doubles the grid by appending its mirror reflection along the axis."""
    arr = to_array(grid)
    if axis == "vertical":
        flipped = np.fliplr(arr)
        return np.concatenate([arr, flipped], axis=1).tolist()
    if axis == "horizontal":
        flipped = np.flipud(arr)
        return np.concatenate([arr, flipped], axis=0).tolist()
    if axis == "both":
        top = np.concatenate([arr, np.fliplr(arr)], axis=1)
        bottom = np.flipud(top)
        return np.concatenate([top, bottom], axis=0).tolist()
    raise ValueError(f"Unknown axis: {axis}")


def complete_symmetry(
    grid: Grid | np.ndarray,
    axis: str = "vertical",
    source: str = "left",
) -> Grid:
    """Completes symmetry by copying one half across the symmetry axis."""
    arr = to_array(grid).copy()
    h, w = arr.shape

    if axis == "vertical":
        mid = w // 2
        for c in range(mid):
            opp = w - 1 - c
            if source == "left":
                arr[:, opp] = arr[:, c]
            else:
                arr[:, c] = arr[:, opp]
    elif axis == "horizontal":
        mid = h // 2
        for r in range(mid):
            opp = h - 1 - r
            if source == "top":
                arr[opp, :] = arr[r, :]
            else:
                arr[r, :] = arr[opp, :]
    return arr.tolist()


def tile_grid(
    grid: Grid | np.ndarray,
    repeats_h: int = 1,
    repeats_w: int = 1,
) -> Grid:
    """Repeats grid periodically along rows and columns."""
    arr = to_array(grid)
    return np.tile(arr, (repeats_h, repeats_w)).tolist()


def replace_color(
    grid: Grid | np.ndarray,
    old_c: int,
    new_c: int,
) -> Grid:
    """Replaces all occurrences of old_c with new_c."""
    arr = to_array(grid).copy()
    arr[arr == old_c] = new_c
    return arr.tolist()


def remap_colors(
    grid: Grid | np.ndarray,
    color_map: dict[int, int],
) -> Grid:
    """Remaps multiple colors simultaneously according to mapping dictionary."""
    arr = to_array(grid).copy()
    out = arr.copy()
    for old_c, new_c in color_map.items():
        out[arr == old_c] = new_c
    return out.tolist()


def color_counts(
    grid: Grid | np.ndarray,
    ignore_background: bool = False,
    background: int = 0,
) -> dict[int, int]:
    """Returns frequency counts of all colors present in the grid."""
    arr = to_array(grid)
    counts = dict(Counter(int(x) for x in arr.flatten()))
    if ignore_background and background in counts:
        del counts[background]
    return counts


def most_frequent_color(
    grid: Grid | np.ndarray,
    ignore_background: bool = True,
    background: int = 0,
) -> int | None:
    """Returns the most frequent color in the grid."""
    counts = color_counts(grid, ignore_background=ignore_background, background=background)
    if not counts:
        return None
    return max(counts.items(), key=lambda x: (x[1], -x[0]))[0]


def least_frequent_color(
    grid: Grid | np.ndarray,
    ignore_background: bool = True,
    background: int = 0,
) -> int | None:
    """Returns the least frequent color in the grid."""
    counts = color_counts(grid, ignore_background=ignore_background, background=background)
    if not counts:
        return None
    return min(counts.items(), key=lambda x: (x[1], x[0]))[0]


def fill_enclosed_regions(
    grid: Grid | np.ndarray,
    fill_color: int,
    background: int = 0,
) -> Grid:
    """Fills background regions that are fully enclosed by non-background pixels."""
    arr = to_array(grid).copy()
    h, w = arr.shape

    # Find all exterior background cells via flood-fill from boundary
    visited = np.zeros((h, w), dtype=bool)
    queue = deque()

    # Seed boundary background cells
    for r in range(h):
        for c in (0, w - 1):
            if arr[r, c] == background and not visited[r, c]:
                visited[r, c] = True
                queue.append((r, c))
    for c in range(w):
        for r in (0, h - 1):
            if arr[r, c] == background and not visited[r, c]:
                visited[r, c] = True
                queue.append((r, c))

    deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    while queue:
        curr_r, curr_c = queue.popleft()
        for dr, dc in deltas:
            nr, nc = curr_r + dr, curr_c + dc
            if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and arr[nr, nc] == background:
                visited[nr, nc] = True
                queue.append((nr, nc))

    # Any background cell not visited from boundary is enclosed
    for r in range(h):
        for c in range(w):
            if arr[r, c] == background and not visited[r, c]:
                arr[r, c] = fill_color

    return arr.tolist()


def crop_largest_object(
    grid: Grid | np.ndarray,
    background: int = 0,
    connectivity: int = 4,
) -> Grid:
    """Finds the largest connected component and crops to its bounding box."""
    objs = find_connected_components(grid, background=background, connectivity=connectivity)
    if not objs:
        return to_grid(grid)
    largest = max(objs, key=lambda o: o.size)
    return largest.crop_from(grid)


def crop_smallest_object(
    grid: Grid | np.ndarray,
    background: int = 0,
    connectivity: int = 4,
) -> Grid:
    """Finds the smallest connected component and crops to its bounding box."""
    objs = find_connected_components(grid, background=background, connectivity=connectivity)
    if not objs:
        return to_grid(grid)
    smallest = min(objs, key=lambda o: o.size)
    return smallest.crop_from(grid)


def remove_small_objects(
    grid: Grid | np.ndarray,
    max_size: int = 1,
    background: int = 0,
    replacement: int = 0,
    connectivity: int = 4,
) -> Grid:
    """Removes noise or small connected components whose size <= max_size."""
    arr = to_array(grid).copy()
    objs = find_connected_components(arr, background=background, connectivity=connectivity)
    for obj in objs:
        if obj.size <= max_size:
            for r, c in obj.coords:
                arr[r, c] = replacement
    return arr.tolist()


def connect_matching_dots(grid: Grid | np.ndarray, background: int = 0) -> Grid:
    """Connects pairs of identical colored dots with orthogonal or diagonal lines."""
    arr = to_array(grid).copy()
    colors = set(np.unique(arr)) - {background}
    for c in colors:
        coords = list(zip(*np.where(arr == c)))
        for i in range(len(coords)):
            for j in range(i + 1, len(coords)):
                r1, c1 = coords[i]
                r2, c2 = coords[j]
                # Same row -> draw horizontal line
                if r1 == r2:
                    start_c, end_c = min(c1, c2), max(c1, c2)
                    path = [arr[r1, k] for k in range(start_c, end_c + 1)]
                    if all(v in (background, c) for v in path):
                        arr[r1, start_c : end_c + 1] = c
                # Same column -> draw vertical line
                elif c1 == c2:
                    start_r, end_r = min(r1, r2), max(r1, r2)
                    path = [arr[k, c1] for k in range(start_r, end_r + 1)]
                    if all(v in (background, c) for v in path):
                        arr[start_r : end_r + 1, c1] = c
                # Diagonal
                elif abs(r1 - r2) == abs(c1 - c2):
                    dr = 1 if r2 > r1 else -1
                    dc = 1 if c2 > c1 else -1
                    length = abs(r1 - r2)
                    path = [arr[r1 + k * dr, c1 + k * dc] for k in range(length + 1)]
                    if all(v in (background, c) for v in path):
                        for k in range(length + 1):
                            arr[r1 + k * dr, c1 + k * dc] = c
    return arr.tolist()


def repair_room_2d_periodic(room_arr: np.ndarray, is_vertical: bool = True) -> np.ndarray:
    """Detects and repairs the dominant 2D periodic motif in a rectangular room."""
    h, w = room_arr.shape
    if h == 0 or w == 0:
        return room_arr

    if is_vertical:
        best_p, best_score, best_block = None, 0.0, None
        for p in range(2, min(9, h // 2 + 1)):
            block = np.zeros((p, w), dtype=int)
            for r in range(p):
                for c in range(w):
                    vals = [room_arr[k, c] for k in range(r, h, p)]
                    block[r, c] = Counter(vals).most_common(1)[0][0]
            cell_matches = sum(
                room_arr[k, c] == block[k % p, c] for k in range(h) for c in range(w)
            )
            score = cell_matches / (h * w) if h * w > 0 else 0.0
            if score > best_score:
                best_score = score
                best_p = p
                best_block = block
        if best_block is not None and best_score >= 0.70:
            repaired = np.zeros_like(room_arr)
            for k in range(h):
                repaired[k, :] = best_block[k % best_p, :]
            return repaired
    else:
        best_p, best_score, best_block = None, 0.0, None
        for p in range(2, min(9, w // 2 + 1)):
            block = np.zeros((h, p), dtype=int)
            for r in range(h):
                for c in range(p):
                    vals = [room_arr[r, k] for k in range(c, w, p)]
                    block[r, c] = Counter(vals).most_common(1)[0][0]
            cell_matches = sum(
                room_arr[r, k] == block[r, k % p] for r in range(h) for k in range(w)
            )
            score = cell_matches / (h * w) if h * w > 0 else 0.0
            if score > best_score:
                best_score = score
                best_p = p
                best_block = block
        if best_block is not None and best_score >= 0.70:
            repaired = np.zeros_like(room_arr)
            for k in range(w):
                repaired[:, k] = best_block[:, k % best_p]
            return repaired

    return room_arr


def repair_periodic_subgrids(grid: Grid | np.ndarray, pad: int = 2) -> Grid:
    """Identifies rooms bounded by grid divider lines and repairs periodic 2D motifs."""
    arr = to_array(grid).copy()
    h, w = arr.shape
    r_divs = (
        [0]
        + [
            r
            for r in range(1, h - 1)
            if len(set(arr[r, :])) <= 2 and Counter(arr[r, :]).most_common(1)[0][1] >= w - 1
        ]
        + [h - 1]
    )
    c_divs = (
        [0]
        + [
            c
            for c in range(1, w - 1)
            if len(set(arr[:, c])) <= 2 and Counter(arr[:, c]).most_common(1)[0][1] >= h - 1
        ]
        + [w - 1]
    )
    r_divs = sorted(list(set(r_divs)))
    c_divs = sorted(list(set(c_divs)))

    out = arr.copy()
    for r_idx in range(len(r_divs) - 1):
        r_start, r_end = r_divs[r_idx], r_divs[r_idx + 1]
        for c_idx in range(len(c_divs) - 1):
            c_start, c_end = c_divs[c_idx], c_divs[c_idx + 1]
            room_h = r_end - r_start + 1
            room_w = c_end - c_start + 1

            if room_h > room_w and room_h >= 2 * pad + 2:
                interior = arr[r_start + pad : r_end - pad + 1, c_start + pad : c_end - pad + 1]
                repaired = repair_room_2d_periodic(interior, is_vertical=True)
                out[r_start + pad : r_end - pad + 1, c_start + pad : c_end - pad + 1] = repaired
            elif room_w >= room_h and room_w >= 2 * pad + 2:
                interior = arr[r_start + pad : r_end - pad + 1, c_start + pad : c_end - pad + 1]
                repaired = repair_room_2d_periodic(interior, is_vertical=False)
                out[r_start + pad : r_end - pad + 1, c_start + pad : c_end - pad + 1] = repaired

    return out.tolist()


def crop_hollow_rectangle_interior(grid: Grid | np.ndarray, background: int = 0) -> Grid:
    """Finds a hollow rectangular frame of uniform color and extracts its interior."""
    arr = to_array(grid)
    for c in sorted(set(np.unique(arr)) - {background}):
        coords = list(zip(*np.where(arr == c)))
        if len(coords) < 8:
            continue
        min_r, min_c = min(r for r, _ in coords), min(col for _, col in coords)
        max_r, max_c = max(r for r, _ in coords), max(col for _, col in coords)
        if max_r - min_r >= 2 and max_c - min_c >= 2:
            top = all(arr[min_r, col] == c for col in range(min_c, max_c + 1))
            bot = all(arr[max_r, col] == c for col in range(min_c, max_c + 1))
            left = all(arr[row, min_c] == c for row in range(min_r, max_r + 1))
            right = all(arr[row, max_c] == c for row in range(min_r, max_r + 1))
            if top and bot and left and right:
                inner = arr[min_r + 1 : max_r, min_c + 1 : max_c]
                if inner.size > 0:
                    return inner.tolist()
    return to_grid(grid)


def select_odd_one_out_block(
    grid: Grid | np.ndarray,
    win_h: int = 3,
    win_w: int = 3,
) -> Grid:
    """Finds composite block components and selects the block with unique symmetry."""
    arr = to_array(grid)
    h, w = arr.shape
    blocks: list[np.ndarray] = []

    if h == win_h and w % win_w == 0:
        for c in range(0, w, win_w):
            blocks.append(arr[:, c : c + win_w])
    elif w == win_w and h % win_h == 0:
        for r in range(0, h, win_h):
            blocks.append(arr[r : r + win_h, :])
    elif h % win_h == 0 and w % win_w == 0:
        for r in range(0, h, win_h):
            for c in range(0, w, win_w):
                blocks.append(arr[r : r + win_h, c : c + win_w])
    else:
        return to_grid(grid)

    if not blocks:
        return to_grid(grid)

    def is_sym(b: np.ndarray) -> bool:
        return bool(np.array_equal(b, np.flipud(b)) and np.array_equal(b, np.fliplr(b)))

    sigs = [is_sym(b) for b in blocks]
    counts = Counter(sigs)
    unique_sigs = [s for s, cnt in counts.items() if cnt == 1]
    if len(unique_sigs) == 1:
        target_idx = sigs.index(unique_sigs[0])
        return blocks[target_idx].tolist()

    return blocks[0].tolist()


def extract_window_relative_to_marker(
    grid: Grid | np.ndarray,
    marker_color: int | None = None,
    relation: str = "below",
    win_h: int = 3,
    win_w: int = 3,
) -> Grid:
    """Extracts a subwindow positioned relative to a single-pixel marker."""
    arr = to_array(grid)
    h, w = arr.shape

    if marker_color is None:
        colors, counts = np.unique(arr, return_counts=True)
        single_colors = [c for c, cnt in zip(colors, counts) if cnt == 1]
        if not single_colors:
            return to_grid(grid)
        marker_color = int(single_colors[0])

    markers = list(zip(*np.where(arr == marker_color)))
    if not markers:
        return to_grid(grid)

    r_m, c_m = markers[0]

    if relation == "below":
        r0, r1 = r_m + 1, r_m + 1 + win_h
        c0, c1 = c_m - win_w // 2, c_m - win_w // 2 + win_w
    elif relation == "above":
        r0, r1 = r_m - win_h, r_m
        c0, c1 = c_m - win_w // 2, c_m - win_w // 2 + win_w
    elif relation == "center":
        r0, r1 = r_m - win_h // 2, r_m - win_h // 2 + win_h
        c0, c1 = c_m - win_w // 2, c_m - win_w // 2 + win_w
    elif relation == "right":
        r0, r1 = r_m - win_h // 2, r_m - win_h // 2 + win_h
        c0, c1 = c_m + 1, c_m + 1 + win_w
    elif relation == "left":
        r0, r1 = r_m - win_h // 2, r_m - win_h // 2 + win_h
        c0, c1 = c_m - win_w, c_m
    else:
        return to_grid(grid)

    if 0 <= r0 and r1 <= h and 0 <= c0 and c1 <= w:
        return arr[r0:r1, c0:c1].tolist()

    return to_grid(grid)


def rigid_object_gravity(
    grid: Grid | np.ndarray,
    direction: str = "down",
    background: int = 0,
) -> Grid:
    """Simulates rigid-body physics: each connected component falls until colliding."""
    arr = to_array(grid).copy()
    h, w = arr.shape
    objs = find_connected_components(arr, background=background, connectivity=8)
    if not objs:
        return to_grid(grid)

    out = np.full((h, w), background, dtype=int)

    if direction == "down":
        objs = sorted(objs, key=lambda o: max(r for r, _ in o.coords), reverse=True)
        for obj in objs:
            coords = list(obj.coords)
            max_shift = 0
            while True:
                can = True
                for r, c in coords:
                    nr = r + max_shift + 1
                    if nr >= h or out[nr, c] != background:
                        can = False
                        break
                if can:
                    max_shift += 1
                else:
                    break
            for r, c in coords:
                out[r + max_shift, c] = arr[r, c]

    elif direction == "up":
        objs = sorted(objs, key=lambda o: min(r for r, _ in o.coords))
        for obj in objs:
            coords = list(obj.coords)
            max_shift = 0
            while True:
                can = True
                for r, c in coords:
                    nr = r - max_shift - 1
                    if nr < 0 or out[nr, c] != background:
                        can = False
                        break
                if can:
                    max_shift += 1
                else:
                    break
            for r, c in coords:
                out[r - max_shift, c] = arr[r, c]

    elif direction == "right":
        objs = sorted(objs, key=lambda o: max(c for _, c in o.coords), reverse=True)
        for obj in objs:
            coords = list(obj.coords)
            max_shift = 0
            while True:
                can = True
                for r, c in coords:
                    nc = c + max_shift + 1
                    if nc >= w or out[r, nc] != background:
                        can = False
                        break
                if can:
                    max_shift += 1
                else:
                    break
            for r, c in coords:
                out[r, c + max_shift] = arr[r, c]

    elif direction == "left":
        objs = sorted(objs, key=lambda o: min(c for _, c in o.coords))
        for obj in objs:
            coords = list(obj.coords)
            max_shift = 0
            while True:
                can = True
                for r, c in coords:
                    nc = c - max_shift - 1
                    if nc < 0 or out[r, nc] != background:
                        can = False
                        break
                if can:
                    max_shift += 1
                else:
                    break
            for r, c in coords:
                out[r, c - max_shift] = arr[r, c]

    return out.tolist()


def raycast_directional(
    grid: Grid | np.ndarray,
    direction: str = "down",
    background: int = 0,
) -> Grid:
    """Projects directional rays from non-background pixels across the grid."""
    arr = to_array(grid).copy()
    h, w = arr.shape
    out = arr.copy()

    dirs: list[tuple[int, int]] = []
    if direction == "down":
        dirs = [(1, 0)]
    elif direction == "up":
        dirs = [(-1, 0)]
    elif direction == "right":
        dirs = [(0, 1)]
    elif direction == "left":
        dirs = [(0, -1)]
    elif direction == "horizontal":
        dirs = [(0, -1), (0, 1)]
    elif direction == "vertical":
        dirs = [(-1, 0), (1, 0)]
    elif direction == "cross":
        dirs = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    else:
        return to_grid(grid)

    coords = list(zip(*np.where(arr != background)))
    if not coords or len(coords) > 25:
        return to_grid(grid)

    for r, c in coords:
        color = arr[r, c]
        for dr, dc in dirs:
            cr, cc = r + dr, c + dc
            while 0 <= cr < h and 0 <= cc < w:
                if out[cr, cc] == background:
                    out[cr, cc] = color
                cr += dr
                cc += dc

    return out.tolist()


def raycast_diagonal_bouncing(
    grid: Grid | np.ndarray,
    background: int = 0,
) -> Grid:
    """Projects 4-way diagonal billiard rays that bounce off grid boundaries."""
    arr = to_array(grid).copy()
    h, w = arr.shape
    out = arr.copy()

    coords = list(zip(*np.where(arr != background)))
    if not coords or len(coords) > 10:
        return to_grid(grid)

    visited_states = set()
    for r, c in coords:
        color = arr[r, c]
        for dr, dc in [(-1, -1), (-1, 1), (1, -1), (1, 1)]:
            curr_r, curr_c = r, c
            curr_dr, curr_dc = dr, dc
            for _ in range(2 * (h + w)):
                state = (curr_r, curr_c, curr_dr, curr_dc)
                if state in visited_states:
                    break
                visited_states.add(state)

                nr = curr_r + curr_dr
                nc = curr_c + curr_dc
                # Bounce on boundary
                if nr < 0 or nr >= h:
                    curr_dr = -curr_dr
                    nr = curr_r + curr_dr
                if nc < 0 or nc >= w:
                    curr_dc = -curr_dc
                    nc = curr_c + curr_dc
                if 0 <= nr < h and 0 <= nc < w:
                    if out[nr, nc] == background:
                        out[nr, nc] = color
                    curr_r, curr_c = nr, nc
                else:
                    break

    return out.tolist()


def extract_object_property_color_rule(
    train_pairs: list[dict[str, Any]],
    property_name: str = "size",
    background: int = 0,
) -> dict[int, int] | None:
    """Finds if a task maps connected components to a color based on size, height, or width."""
    prop_map: dict[int, int] = {}
    for pair in train_pairs:
        inp = pair["input"]
        out = np.asarray(pair["output"], dtype=int)
        inp_arr = np.asarray(inp, dtype=int)
        if inp_arr.shape != out.shape:
            return None

        objs = find_connected_components(inp, background=background, connectivity=8)
        for obj in objs:
            out_colors = {int(out[r, c]) for r, c in obj.coords}
            if len(out_colors) != 1:
                return None
            out_color = out_colors.pop()
            val = getattr(obj, property_name)
            if val in prop_map and prop_map[val] != out_color:
                return None
            prop_map[val] = out_color

    return prop_map if len(prop_map) >= 2 else None


def color_objects_by_property(
    grid: Grid | np.ndarray,
    property_map: dict[int, int],
    property_name: str = "size",
    background: int = 0,
) -> Grid:
    """Recolors each connected component according to its size, height, or width."""
    arr = to_array(grid).copy()
    objs = find_connected_components(grid, background=background, connectivity=8)
    for obj in objs:
        val = getattr(obj, property_name)
        if val in property_map:
            new_c = property_map[val]
            for r, c in obj.coords:
                arr[r, c] = new_c
    return arr.tolist()








from typing import Any
import numpy as np



def split_and_overlay(grid: Grid | np.ndarray, op: str, background: int = 0) -> Grid | None:
    """Splits a grid into halves (across a divider or midpoint) and applies logical overlay."""
    arr = np.asarray(grid, dtype=int)
    h, w = arr.shape

    g1: np.ndarray | None = None
    g2: np.ndarray | None = None

    if op.startswith("vsplit"):
        if w % 2 == 1 and "_div" in op:
            mid = w // 2
            g1 = arr[:, :mid]
            g2 = arr[:, mid + 1 :]
        elif w % 2 == 0 and "_div" not in op:
            mid = w // 2
            g1 = arr[:, :mid]
            g2 = arr[:, mid:]
        if g1 is None or g2 is None or g1.shape != g2.shape:
            return None
        if "mirror" in op:
            g2 = np.fliplr(g2)
    elif op.startswith("hsplit"):
        if h % 2 == 1 and "_div" in op:
            mid = h // 2
            g1 = arr[:mid, :]
            g2 = arr[mid + 1 :, :]
        elif h % 2 == 0 and "_div" not in op:
            mid = h // 2
            g1 = arr[:mid, :]
            g2 = arr[mid:, :]
        if g1 is None or g2 is None or g1.shape != g2.shape:
            return None
        if "mirror" in op:
            g2 = np.flipud(g2)
    else:
        return None

    if g1 is None or g2 is None:
        return None

    if op.endswith("_or"):
        res = np.where(g1 != background, g1, g2)
    elif op.endswith("_and"):
        res = np.where((g1 != background) & (g2 != background), g1, background)
    elif op.endswith("_xor"):
        res = np.where(
            (g1 != background) ^ (g2 != background),
            np.where(g1 != background, g1, g2),
            background,
        )
    else:
        return None

    return res.tolist()


SPLIT_OVERLAY_OPERATIONS = [
    "vsplit_div_or",
    "vsplit_div_and",
    "vsplit_div_xor",
    "vsplit_or",
    "vsplit_and",
    "vsplit_xor",
    "vsplit_mirror_or",
    "vsplit_mirror_and",
    "vsplit_mirror_xor",
    "hsplit_div_or",
    "hsplit_div_and",
    "hsplit_div_xor",
    "hsplit_or",
    "hsplit_and",
    "hsplit_xor",
    "hsplit_mirror_or",
    "hsplit_mirror_and",
    "hsplit_mirror_xor",
]


from typing import Any, Sequence
import numpy as np



def extract_3x3_local_rules(
    train_pairs: list[dict[str, Any]],
) -> dict[tuple[int, ...], int] | None:
    """Extracts deterministic 3x3 local neighborhood transition rules.

    Maps a flattened 9-tuple representing a 3x3 window around cell (r, c)
    to the target color at (r, c). Returns None if there is any contradictory
    mapping (i.e. identical 3x3 context maps to different target colors).
    """
    rules: dict[tuple[int, ...], int] = {}

    for pair in train_pairs:
        inp = np.asarray(pair["input"], dtype=int)
        out = np.asarray(pair["output"], dtype=int)
        if inp.shape != out.shape:
            return None

        h, w = inp.shape
        # Pad with -1 representing boundary / out of bounds
        padded = np.pad(inp, 1, mode="constant", constant_values=-1)

        for r in range(h):
            for c in range(w):
                patch = tuple(padded[r : r + 3, c : c + 3].flatten())
                target = int(out[r, c])
                if patch in rules and rules[patch] != target:
                    return None
                rules[patch] = target

    return rules if rules else None


def apply_3x3_local_rules(
    grid: Grid | np.ndarray,
    rules: dict[tuple[int, ...], int],
    fallback_to_input: bool = True,
) -> Grid | None:
    """Applies 3x3 local neighborhood transition rules to an input grid.

    If fallback_to_input is False, returns None if any patch was unseen.
    """
    arr = np.asarray(grid, dtype=int)
    h, w = arr.shape
    padded = np.pad(arr, 1, mode="constant", constant_values=-1)
    out = np.zeros_like(arr)

    for r in range(h):
        for c in range(w):
            patch = tuple(padded[r : r + 3, c : c + 3].flatten())
            if patch in rules:
                out[r, c] = rules[patch]
            elif fallback_to_input:
                out[r, c] = arr[r, c]
            else:
                return None

    return out.tolist()


from dataclasses import dataclass
from typing import Any, Callable, Sequence
import numpy as np





@dataclass
class Program:
    """Executable DSL program composed of one or more primitive functions."""

    name: str
    steps: list[tuple[str, Callable[[Grid], Grid]]]

    def execute(self, grid: Grid) -> Grid:
        curr = grid
        for _, fn in self.steps:
            curr = fn(curr)
        return curr


def find_dynamic_color_mapping(train_pairs: list[dict[str, Any]]) -> dict[int, int] | None:
    """Finds if a consistent global color remap explains the input->output pairs."""
    mapping: dict[int, int] = {}
    for pair in train_pairs:
        inp = np.asarray(pair["input"], dtype=int)
        out = np.asarray(pair["output"], dtype=int)
        if inp.shape != out.shape:
            return None
        for in_v, out_v in zip(inp.flat, out.flat):
            if in_v in mapping and mapping[in_v] != out_v:
                return None
            mapping[in_v] = int(out_v)
    return mapping if mapping else None


def generate_single_step_candidates(train_pairs: list[dict[str, Any]]) -> list[Program]:
    """Generates 1-step candidate transformation programs."""
    progs: list[Program] = []

    # 1. Fixed geometric transforms
    geom_ops: list[tuple[str, Callable[[Grid], Grid]]] = [
        ("identity", lambda g: [row[:] for row in g]),
        ("rot90", rotate_90),
        ("rot180", rotate_180),
        ("rot270", rotate_270),
        ("reflect_h", reflect_horizontal),
        ("reflect_v", reflect_vertical),
        ("transpose", transpose),
    ]
    for name, op in geom_ops:
        progs.append(Program(name=name, steps=[(name, op)]))

    # 2. Gravity drops along cardinal directions
    for direction in ["down", "up", "left", "right"]:
        op_name = f"gravity_{direction}"
        progs.append(
            Program(
                name=op_name,
                steps=[(op_name, lambda g, d=direction: gravity(g, direction=d))],
            )
        )

    # 3. Content cropping for various background colors
    # Find background colors present in inputs
    bg_candidates = {0}
    for p in train_pairs:
        inp_arr = np.asarray(p["input"], dtype=int)
        bg_candidates.add(int(inp_arr[0, 0]))

    for bg in sorted(bg_candidates):
        crop_name = f"crop_content_bg{bg}"
        progs.append(
            Program(
                name=crop_name,
                steps=[(crop_name, lambda g, b=bg: crop_to_content(g, background=b))],
            )
        )
        # Crop largest / smallest object
        for conn in [4, 8]:
            p_lg = f"crop_largest_bg{bg}_c{conn}"
            progs.append(
                Program(
                    name=p_lg,
                    steps=[(p_lg, lambda g, b=bg, c=conn: crop_largest_object(g, background=b, connectivity=c))],
                )
            )
            p_sm = f"crop_smallest_bg{bg}_c{conn}"
            progs.append(
                Program(
                    name=p_sm,
                    steps=[(p_sm, lambda g, b=bg, c=conn: crop_smallest_object(g, background=b, connectivity=c))],
                )
            )
        # Noise removal
        p_noise = f"remove_noise_bg{bg}"
        progs.append(
            Program(
                name=p_noise,
                steps=[(p_noise, lambda g, b=bg: remove_small_objects(g, max_size=1, background=b))],
            )
        )
        # Rigid body object gravity
        for direction in ["down", "up", "left", "right"]:
            p_rg = f"rigid_gravity_{direction}_bg{bg}"
            progs.append(
                Program(
                    name=p_rg,
                    steps=[(p_rg, lambda g, d=direction, b=bg: rigid_object_gravity(g, direction=d, background=b))],
                )
            )

    # Enclosed region filling
    output_colors = set()
    for p in train_pairs:
        output_colors.update(np.unique(np.asarray(p["output"])).tolist())
    for fill_c in sorted(output_colors):
        for bg in sorted(bg_candidates):
            if fill_c != bg:
                enc_name = f"fill_enclosed_bg{bg}_c{fill_c}"
                progs.append(
                    Program(
                        name=enc_name,
                        steps=[(enc_name, lambda g, b=bg, fc=fill_c: fill_enclosed_regions(g, fill_color=fc, background=b))],
                    )
                )

    # 4. Mirror appending
    for axis in ["vertical", "horizontal", "both"]:
        mirr_name = f"mirror_append_{axis}"
        progs.append(
            Program(
                name=mirr_name,
                steps=[(mirr_name, lambda g, a=axis: mirror_append(g, axis=a))],
            )
        )

    # 4b. Connect matching dots
    for bg in sorted(bg_candidates):
        cmd_name = f"connect_dots_bg{bg}"
        progs.append(
            Program(
                name=cmd_name,
                steps=[(cmd_name, lambda g, b=bg: connect_matching_dots(g, background=b))],
            )
        )

    # 4c. Repair periodic rooms / subgrids
    for pad in [1, 2]:
        per_name = f"repair_periodic_pad{pad}"
        progs.append(
            Program(
                name=per_name,
                steps=[(per_name, lambda g, p=pad: repair_periodic_subgrids(g, pad=p))],
            )
        )

    # 4d. Crop hollow rectangle interior
    for bg in sorted(bg_candidates):
        hr_name = f"crop_hollow_rect_bg{bg}"
        progs.append(
            Program(
                name=hr_name,
                steps=[(hr_name, lambda g, b=bg: crop_hollow_rectangle_interior(g, background=b))],
            )
        )

    # 5. Symmetry completions
    for axis in ["vertical", "horizontal"]:
        for src in (["left", "right"] if axis == "vertical" else ["top", "bottom"]):
            sym_name = f"complete_sym_{axis}_{src}"
            progs.append(
                Program(
                    name=sym_name,
                    steps=[
                        (
                            sym_name,
                            lambda g, a=axis, s=src: complete_symmetry(g, axis=a, source=s),
                        )
                    ],
                )
            )

    # 6. Integer scaling (factors 2, 3)
    for factor in [2, 3]:
        scale_name = f"scale_{factor}"
        progs.append(
            Program(
                name=scale_name,
                steps=[(scale_name, lambda g, f=factor: scale_grid(g, factor=f))],
            )
        )

    # 7. Tiling (2x2, 2x1, 1x2, 3x3)
    for rh, rw in [(2, 2), (2, 1), (1, 2), (3, 3)]:
        tile_name = f"tile_{rh}x{rw}"
        progs.append(
            Program(
                name=tile_name,
                steps=[(tile_name, lambda g, h=rh, w=rw: tile_grid(g, repeats_h=h, repeats_w=w))],
            )
        )

    # 8. Dynamic consistent color mapping
    cmap = find_dynamic_color_mapping(train_pairs)
    if cmap is not None:
        cmap_name = f"dynamic_remap_colors"
        progs.append(
            Program(
                name=cmap_name,
                steps=[(cmap_name, lambda g, m=cmap: remap_colors(g, m))],
            )
        )

    # 9. Split and Logical Overlay
    for op in SPLIT_OVERLAY_OPERATIONS:
        op_name = f"overlay_{op}"
        progs.append(
            Program(
                name=op_name,
                steps=[(op_name, lambda g, o=op: split_and_overlay(g, op=o) or g)],
            )
        )

    # 10. Cellular Automata 3x3 Local Rules
    rules_3x3 = extract_3x3_local_rules(train_pairs)
    if rules_3x3 is not None:
        ca_name = "cellular_automata_3x3"
        progs.append(
            Program(
                name=ca_name,
                steps=[(ca_name, lambda g, r=rules_3x3: apply_3x3_local_rules(g, r, fallback_to_input=True) or g)],
            )
        )

    # 11. Odd-one-out 3x3 Block Selection
    progs.append(
        Program(
            name="select_odd_one_out_3x3",
            steps=[("select_odd_one_out_3x3", lambda g: select_odd_one_out_block(g, win_h=3, win_w=3))],
        )
    )

    # 12. Window Relative to Single Marker
    for rel in ["below", "above", "center", "right", "left"]:
        m_name = f"marker_win_{rel}_3x3"
        progs.append(
            Program(
                name=m_name,
                steps=[(m_name, lambda g, r=rel: extract_window_relative_to_marker(g, relation=r, win_h=3, win_w=3))],
            )
        )

    # 13. Directional and Diagonal Raycasting
    for d in ["down", "up", "right", "left", "vertical", "horizontal", "cross"]:
        rc_name = f"raycast_{d}"
        progs.append(
            Program(
                name=rc_name,
                steps=[(rc_name, lambda g, d_dir=d: raycast_directional(g, direction=d_dir))],
            )
        )
    progs.append(
        Program(
            name="raycast_diagonal_bouncing",
            steps=[("raycast_diagonal_bouncing", lambda g: raycast_diagonal_bouncing(g))],
        )
    )

    # 14. Color Objects by Size, Height, or Width
    for prop in ["size", "height", "width"]:
        prop_rule = extract_object_property_color_rule(train_pairs, property_name=prop)
        if prop_rule is not None:
            p_name = f"color_objs_by_{prop}"
            progs.append(
                Program(
                    name=p_name,
                    steps=[(p_name, lambda g, pr=prop, pm=prop_rule: color_objects_by_property(g, pm, property_name=pr))],
                )
            )

    return progs


def generate_two_step_candidates(
    train_pairs: list[dict[str, Any]],
    promising_step1: list[Program],
) -> list[Program]:
    """Generates 2-step pipelines combining promising transformations."""
    progs: list[Program] = []

    # Target output sizes across train pairs
    target_shapes = {np.asarray(p["output"]).shape for p in train_pairs}

    # Step 2 candidate pool
    step2_pool: list[tuple[str, Callable[[Grid], Grid]]] = [
        ("rot90", rotate_90),
        ("rot180", rotate_180),
        ("rot270", rotate_270),
        ("reflect_h", reflect_horizontal),
        ("reflect_v", reflect_vertical),
        ("transpose", transpose),
    ]

    cmap = find_dynamic_color_mapping(train_pairs)
    if cmap is not None:
        step2_pool.append(("remap_colors", lambda g, m=cmap: remap_colors(g, m)))

    for p1 in promising_step1:
        # Check if output dimension of p1 matches target shapes on first train pair
        try:
            sample_out = p1.execute(train_pairs[0]["input"])
            sample_shape = (len(sample_out), len(sample_out[0]) if sample_out else 0)
        except Exception:
            continue

        for s2_name, s2_fn in step2_pool:
            composite_name = f"{p1.name} >> {s2_name}"
            progs.append(
                Program(
                    name=composite_name,
                    steps=list(p1.steps) + [(s2_name, s2_fn)],
                )
            )

    return progs


def evaluate_program(
    program: Program,
    train_pairs: list[dict[str, Any]],
    short_circuit: bool = True,
) -> float:
    """Evaluates program on training demonstration pairs.

    Returns fraction of training pairs perfectly solved (0.0 to 1.0).
    """
    if not train_pairs:
        return 0.0

    solved_count = 0
    for pair in train_pairs:
        try:
            pred = program.execute(pair["input"])
            if grid_equal(pred, pair["output"]):
                solved_count += 1
            elif short_circuit:
                return 0.0
        except Exception:
            if short_circuit:
                return 0.0

    return solved_count / len(train_pairs)


def search_dsl_programs(
    train_pairs: list[dict[str, Any]],
    max_level: int = 2,
    max_perfect_programs: int = 5,
) -> list[tuple[Program, float]]:
    """Searches for DSL programs solving 100% of demonstration pairs.

    Returns:
        List of (program, train_score) tuples, sorted by simplicity.
    """
    perfect_programs: list[tuple[Program, float]] = []

    # Level 1 Search
    level1_candidates = generate_single_step_candidates(train_pairs)
    promising_for_step2: list[Program] = []

    for prog in level1_candidates:
        score = evaluate_program(prog, train_pairs, short_circuit=True)
        if score == 1.0:
            perfect_programs.append((prog, 1.0))
            if len(perfect_programs) >= max_perfect_programs:
                return perfect_programs
        else:
            # Check if dimension matches
            try:
                out0 = prog.execute(train_pairs[0]["input"])
                target0 = train_pairs[0]["output"]
                if len(out0) == len(target0) and len(out0[0]) == len(target0[0]):
                    promising_for_step2.append(prog)
            except Exception:
                pass

    if max_level >= 2 and len(perfect_programs) < max_perfect_programs:
        # Level 2 Search
        level2_candidates = generate_two_step_candidates(
            train_pairs, promising_for_step2[:15]
        )
        for prog in level2_candidates:
            score = evaluate_program(prog, train_pairs, short_circuit=True)
            if score == 1.0:
                perfect_programs.append((prog, 1.0))
                if len(perfect_programs) >= max_perfect_programs:
                    break

    return perfect_programs


from typing import Any
import numpy as np




class DSLSolver:
    """End-to-end ARC puzzle solver using symbolic DSL search and robust fallbacks."""

    def __init__(self, max_level: int = 2) -> None:
        self.max_level = max_level

    def solve_task(self, task: dict[str, Any]) -> list[dict[str, Grid]]:
        """Solves a single ARC task across all its test pairs."""
        train_pairs = task.get("train", [])
        test_pairs = task.get("test", [])

        # 1. Search for programs solving 100% of demonstration pairs
        perfect_programs = search_dsl_programs(
            train_pairs, max_level=self.max_level, max_perfect_programs=4
        )

        results: list[dict[str, Grid]] = []

        for test_item in test_pairs:
            test_in = test_item["input"]
            attempt_1: Grid | None = None
            attempt_2: Grid | None = None

            # Try generating attempt_1 from top program
            if perfect_programs:
                best_prog, _ = perfect_programs[0]
                try:
                    cand = best_prog.execute(test_in)
                    if len(cand) > 0 and len(cand[0]) > 0:
                        attempt_1 = cand
                except Exception:
                    attempt_1 = None

            # Try generating attempt_2 from subsequent distinct program
            if perfect_programs and len(perfect_programs) > 1:
                for prog, _ in perfect_programs[1:]:
                    try:
                        cand = prog.execute(test_in)
                        if len(cand) > 0 and len(cand[0]) > 0:
                            if attempt_1 is None or not grid_equal(cand, attempt_1):
                                attempt_2 = cand
                                break
                    except Exception:
                        continue

            # Fallbacks if attempts are still missing
            if attempt_1 is None:
                attempt_1 = [row[:] for row in test_in]

            if attempt_2 is None:
                # Try horizontal reflection
                alt = reflect_horizontal(test_in)
                if not grid_equal(alt, attempt_1):
                    attempt_2 = alt
                else:
                    # Try vertical reflection
                    alt2 = reflect_vertical(test_in)
                    if not grid_equal(alt2, attempt_1):
                        attempt_2 = alt2
                    else:
                        attempt_2 = [row[:] for row in test_in]

            results.append({"attempt_1": attempt_1, "attempt_2": attempt_2})

        return results

    def predict_all(self, challenges: dict[str, Any]) -> dict[str, list[dict[str, Grid]]]:
        """Runs the solver on all tasks in the challenges dictionary."""
        preds: dict[str, list[dict[str, Grid]]] = {}
        for task_id, task in challenges.items():
            preds[task_id] = self.solve_task(task)
        return preds


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

In [ ]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=12 python starter.py --end-time {global_end_time}

In [ ]:
import os
import json
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder
import arc_dsl

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    if not os.path.exists(test_path):
        test_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    data = ArcDataset.from_file(test_path)
else:
    eval_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    if not os.path.exists(eval_path):
        eval_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
    data = ArcDataset.from_file(eval_path)
    sol_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
    if not os.path.exists(sol_path):
        sol_path = "/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"
    data = data.load_replies(sol_path)

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results("/kaggle/inference_outputs")

# Base neural predictions
submission = data.get_submission(decoder.run_selection_algo())

# Tier 1 Symbolic Induction Filter
print("*** Running Tier 1 Symbolic Verification pre-pass...")
symbolic_solver = arc_dsl.DSLSolver(max_level=2)
tier1_hits = 0

for task_id, task in data.queries.items():
    train_pairs = task.get("train", [])
    perfect_progs = arc_dsl.search_dsl_programs(train_pairs, max_level=2, max_perfect_programs=2)
    if perfect_progs:
        best_name = perfect_progs[0][0].name
        print(f"[Tier 1 Verified] Task {task_id}: {best_name}")
        sym_attempts = symbolic_solver.solve_task(task)
        if task_id in submission and len(submission[task_id]) == len(sym_attempts):
            tier1_hits += 1
            for i, sym_item in enumerate(sym_attempts):
                base_cand_1 = submission[task_id][i].get("attempt_1")
                base_cand_2 = submission[task_id][i].get("attempt_2")
                sym_cand = sym_item["attempt_1"]
                # Optimal 2-attempt union: keep top neural guess if distinct from symbolic
                submission[task_id][i]["attempt_1"] = sym_cand
                if arc_dsl.grid_equal(sym_cand, base_cand_1):
                    submission[task_id][i]["attempt_2"] = base_cand_2
                else:
                    submission[task_id][i]["attempt_2"] = base_cand_1

print(f"*** Tier 1 Symbolic Verified Hits: {tier1_hits}/{len(data.queries)} tasks")

with open("submission.json", "w") as f:
    json.dump(submission, f)

with open("submission.json", "r") as f:
    reload_submission = json.load(f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    print("*** Final Hybrid Validation score:", data.validate_submission(reload_submission))
